# TP Individual — QuéLibroLeo
### E72.1.01 · Fundamentos de Métodos Analíticos Predictivos

**Problema.** Predecir si a un lector le va a gustar un libro que no leyó, a partir de
los datos de quelibroleo.com. Es una **clasificación binaria**: `rating >= 7` es que le
gustó (`1`), `rating <= 5` que no (`0`), y los `rating == 6` se descartan por ser un
valor gris que no aporta información.

**Métrica de decisión: `f1`.** El dataset está desbalanceado (~84/16), así que `accuracy`
no sirve para decidir nada. Todas las comparaciones entre experimentos usan `f1` y sólo
`f1`; la matriz de confusión se muestra como material descriptivo.

**Semilla: 42** en todo (split, modelos, samplers, muestreos), para que los errores sean
comparables entre corridas.

> Este notebook se genera automáticamente desde los módulos de `src/` con
> `python -m src.armar_notebook`. No editarlo a mano: los cambios se hacen en los `.py`.


## 0. Los datos

Los tres CSV no están en el repositorio. En Colab, montá el Drive y apuntá
`QLL_DATA_DIR` a la carpeta que los contiene. Si corrés local con los CSV al lado del
notebook, no hace falta tocar nada.


In [1]:
import os

# --- Colab: descomentar estas dos líneas y ajustar la ruta ---
# from google.colab import drive; drive.mount("/content/drive")
# os.environ["QLL_DATA_DIR"] = "/content/drive/MyDrive/TP_QueLibroLeo/data"

import pandas as pd

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 50)


## 1. Configuración

Configuración global del TP: rutas, semilla, métrica de decisión y cortes de rating.

Todo lo que otro módulo necesite parametrizar vive acá. Nadie más hardcodea rutas
ni números mágicos.

*(desde `src/config.py`)*


In [2]:
import os
from pathlib import Path

# --------------------------------------------------------------------------------------
# Rutas
# --------------------------------------------------------------------------------------

def _raiz() -> Path:
    """Raíz del proyecto, tanto corriendo como módulo como dentro del notebook.

    En el notebook no existe `__file__` y el directorio de trabajo puede ser
    `notebooks/`, así que se sube hasta encontrar el CLAUDE.md. En Colab, donde no
    está, cae en el directorio de trabajo y todo cuelga de ahí.
    """
    if "__file__" in globals():
        return Path(__file__).resolve().parent.parent
    actual = Path.cwd().resolve()
    for candidata in (actual, *actual.parents):
        if (candidata / "CLAUDE.md").is_file():
            return candidata
    return actual


RAIZ = _raiz()

# Los CSV no están versionados. Por defecto se buscan en data/; se puede apuntar a
# otro lado con la variable de entorno QLL_DATA_DIR (útil en Colab, donde los datos
# quedan montados en Drive).
DIR_DATOS = Path(os.environ.get("QLL_DATA_DIR", RAIZ / "data"))
DIR_CHECKPOINTS = Path(os.environ.get("QLL_CHECKPOINTS_DIR", RAIZ / "checkpoints"))
DIR_RESULTADOS = RAIZ / "resultados"
DIR_FIGURAS = DIR_RESULTADOS / "figuras"

# Nombres de archivo tal como los entregó la cátedra. La tabla de opiniones viene
# con el nombre "interacciones.csv".
ARCHIVO_LIBROS = os.environ.get("QLL_CSV_LIBROS", "libros.csv")
ARCHIVO_LECTORES = os.environ.get("QLL_CSV_LECTORES", "lectores.csv")
ARCHIVO_OPINIONES = os.environ.get("QLL_CSV_OPINIONES", "interacciones.csv")


def resolver_csv(nombre: str) -> Path:
    """Devuelve la ruta del CSV: primero en DIR_DATOS, si no está, en la raíz del repo.

    El fallback a la raíz existe porque los CSV de la cátedra hoy están ahí; la ruta
    canónica sigue siendo data/. Si no aparece en ninguna de las dos, devuelve la
    canónica para que el error de lectura indique dónde había que ponerlo.
    """
    for candidata in (DIR_DATOS / nombre, RAIZ / nombre):
        if candidata.is_file():
            return candidata
    return DIR_DATOS / nombre


CSV_LIBROS = resolver_csv(ARCHIVO_LIBROS)
CSV_LECTORES = resolver_csv(ARCHIVO_LECTORES)
CSV_OPINIONES = resolver_csv(ARCHIVO_OPINIONES)

CHECKPOINT_BASE = DIR_CHECKPOINTS / "01_base.pkl"

# --------------------------------------------------------------------------------------
# Reproducibilidad y evaluación
# --------------------------------------------------------------------------------------

SEED = 42

# Métrica única de decisión (CLAUDE.md 3.2). El dataset está desbalanceado ~80/20,
# así que accuracy está prohibida. Todas las comparaciones entre experimentos se
# hacen con esta y sólo con esta.
METRICA = "f1"

# --------------------------------------------------------------------------------------
# Definición del target
# --------------------------------------------------------------------------------------

COL_RATING = "rating"
TARGET = "gusto"

RATING_MIN_GUSTO = 7      # rating >= 7  -> gusto = 1
RATING_MAX_NO_GUSTO = 5   # rating <= 5  -> gusto = 0
RATING_DESCARTADO = 6     # rating == 6  -> la fila se elimina (rating gris)

# --------------------------------------------------------------------------------------
# Unión de tablas
# --------------------------------------------------------------------------------------

COL_ID_LIBRO = "id_libro"
COL_ID_LECTOR = "id_lector"

# Cómo se resuelven las opiniones que apuntan a un libro o a un lector inexistente.
# Se decide con el diagnóstico de src/diagnostico.py.
HOW_UNION = "left"


En los `.py` el código está repartido en módulos y se referencia como `config.SEED` o
`carga.unir(...)`. En el notebook es todo un mismo espacio de nombres, así que estos alias
hacen que esas referencias sigan funcionando sin tener que reescribir una sola línea.


In [3]:
import sys

config = carga = sys.modules["__main__"]


## 2. Carga y unión de las tres tablas

Lectura y unión de las tres tablas. Acá no se limpia nada: sólo se lee, se une
y se construye el target.

*(desde `src/carga.py`)*


In [4]:
import pandas as pd



# --------------------------------------------------------------------------------------
# Lectura
# --------------------------------------------------------------------------------------

def cargar_tablas() -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """Lee los tres CSV tal cual vienen y los devuelve por separado, sin transformar.

    No se castean tipos ni se parsean fechas a propósito: cualquier conversión es una
    decisión de limpieza y va en la rama que corresponde.
    """
    libros = pd.read_csv(config.CSV_LIBROS, low_memory=False)
    lectores = pd.read_csv(config.CSV_LECTORES, low_memory=False)
    opiniones = pd.read_csv(config.CSV_OPINIONES, low_memory=False)
    return libros, lectores, opiniones


# --------------------------------------------------------------------------------------
# Perfilado
# --------------------------------------------------------------------------------------

def porcentaje_nulos(df: pd.DataFrame) -> pd.Series:
    """% de nulos por columna, ordenado de mayor a menor."""
    return (df.isna().mean() * 100).round(2).sort_values(ascending=False)


def perfilar_tablas(
    libros: pd.DataFrame,
    lectores: pd.DataFrame,
    opiniones: pd.DataFrame,
) -> None:
    """Imprime filas, tipos, % de nulos por columna e IDs únicos de cada tabla."""
    # Para cada tabla: sus columnas de ID, y la clave que debería identificar una fila.
    tablas = [
        ("libros", libros, [config.COL_ID_LIBRO], [config.COL_ID_LIBRO]),
        ("lectores", lectores, [config.COL_ID_LECTOR], [config.COL_ID_LECTOR]),
        ("opiniones", opiniones, [config.COL_ID_LECTOR, config.COL_ID_LIBRO],
         [config.COL_ID_LECTOR, config.COL_ID_LIBRO]),
    ]

    for nombre, df, cols_id, clave in tablas:
        print(f"\n### Tabla `{nombre}` — {len(df):,} filas × {df.shape[1]} columnas")

        perfil = pd.DataFrame(
            {
                "tipo": df.dtypes.astype(str),
                "% nulos": (df.isna().mean() * 100).round(2),
                "valores únicos": df.nunique(),
            }
        )
        print(perfil.to_string())

        for col in cols_id:
            print(f"  IDs únicos en `{col}`: {df[col].nunique():,}")
        duplicados = int(df.duplicated(subset=clave).sum())
        print(f"  Filas duplicadas por la clave {clave}: {duplicados:,}"
              f"{'  <-- el merge podría multiplicar filas' if duplicados else ''}")


# --------------------------------------------------------------------------------------
# Unión
# --------------------------------------------------------------------------------------

def unir(
    libros: pd.DataFrame,
    lectores: pd.DataFrame,
    opiniones: pd.DataFrame,
    how: str = config.HOW_UNION,
) -> pd.DataFrame:
    """Une las tres tablas dejando a `opiniones` en el centro: una fila = una opinión.

    `how` decide qué pasa con una opinión cuyo libro o lector no existe en las otras
    tablas: con "left" la fila se conserva con todos los atributos en nulo, con "inner"
    se descarta. Ninguna de las dos tablas laterales tiene IDs duplicados, así que el
    merge no puede multiplicar filas.

    Las columnas `genero` de libros y de lectores colisionan: quedan como
    `genero_libro` y `genero_lector`.
    """
    return (
        opiniones
        .merge(libros, on=config.COL_ID_LIBRO, how=how)
        .merge(lectores, on=config.COL_ID_LECTOR, how=how, suffixes=("_libro", "_lector"))
    )


# --------------------------------------------------------------------------------------
# Target
# --------------------------------------------------------------------------------------

def construir_target(df: pd.DataFrame) -> pd.DataFrame:
    """Descarta los rating 6 y recién después mapea >= 7 a 1 y <= 5 a 0.

    El orden importa: el 6 es un rating gris que no aporta información, así que sale
    del dataset antes de binarizar. Se conserva la columna `rating` para trazabilidad,
    pero queda prohibida como predictora (CLAUDE.md 3.3): contiene el target.
    """
    rating = df[config.COL_RATING]
    mask = rating.notna() & rating.ne(config.RATING_DESCARTADO)

    return (
        df.loc[mask]
        .assign(**{config.TARGET: (rating[mask] >= config.RATING_MIN_GUSTO).astype("int8")})
        .reset_index(drop=True)
    )


## 3. Diagnóstico de la unión

Diagnóstico de la unión de las tres tablas. Material para el informe.

Corre la carga, el perfilado, la unión y la construcción del target, e imprime:
cobertura de libros y lectores, opiniones huérfanas, % de nulos antes y después
del merge, comparación left vs inner y distribución de clases.

Uso:  python -m src.diagnostico [--how left|inner]
      python -m src.diagnostico | tee resultados/01_diagnostico_union.txt

*(desde `src/diagnostico.py`)*


In [5]:
import pandas as pd


COLS_LIBROS = ["titulo", "autor", "genero_libro", "editorial", "anio_edicion",
               "isbn", "resumen", "img_src"]
COLS_LECTORES = ["nombre", "genero_lector", "vive_en", "nacimiento"]
COLS_OPINIONES = ["id_lector", "id_libro", "fecha", "rating"]

# Nombre en la tabla de origen -> nombre después del merge (la colisión de `genero`).
RENOMBRES = {"genero_libro": "genero", "genero_lector": "genero"}


def titulo(texto: str) -> None:
    print("\n" + "=" * 88)
    print(texto)
    print("=" * 88)


# --------------------------------------------------------------------------------------
# 1. Cobertura: cuánto de cada tabla lateral usa realmente el dataset
# --------------------------------------------------------------------------------------

def cobertura(libros: pd.DataFrame, lectores: pd.DataFrame, opiniones: pd.DataFrame) -> pd.DataFrame:
    """Cuántas entidades hay en cada tabla y cuántas aparecen en al menos una opinión."""
    filas = []
    for nombre, df, col in [("libros", libros, config.COL_ID_LIBRO),
                            ("lectores", lectores, config.COL_ID_LECTOR)]:
        en_tabla = df[col].nunique()
        referenciados = set(opiniones[col].unique())
        con_opinion = df[col].isin(referenciados).sum()
        filas.append({
            "tabla": nombre,
            "en la tabla": en_tabla,
            "con al menos una opinión": int(con_opinion),
            "% usado": round(100 * con_opinion / en_tabla, 2),
            "sin ninguna opinión": int(en_tabla - con_opinion),
            "% descartado por el merge": round(100 * (en_tabla - con_opinion) / en_tabla, 2),
        })
    return pd.DataFrame(filas).set_index("tabla")


def huerfanas(libros: pd.DataFrame, lectores: pd.DataFrame, opiniones: pd.DataFrame) -> pd.DataFrame:
    """Opiniones que apuntan a un libro o a un lector que no existe en su tabla."""
    sin_libro = ~opiniones[config.COL_ID_LIBRO].isin(set(libros[config.COL_ID_LIBRO]))
    sin_lector = ~opiniones[config.COL_ID_LECTOR].isin(set(lectores[config.COL_ID_LECTOR]))
    total = len(opiniones)

    filas = [
        ("libro inexistente", int(sin_libro.sum())),
        ("lector inexistente", int(sin_lector.sum())),
        ("libro Y lector inexistentes", int((sin_libro & sin_lector).sum())),
        ("al menos una de las dos (se pierden con inner)", int((sin_libro | sin_lector).sum())),
    ]
    return pd.DataFrame(
        [{"caso": c, "opiniones": n, "% del total": round(100 * n / total, 3)} for c, n in filas]
    ).set_index("caso")


# --------------------------------------------------------------------------------------
# 2. Nulos antes y después del merge
# --------------------------------------------------------------------------------------

def tabla_nulos(libros, lectores, opiniones, unidos: dict[str, pd.DataFrame]) -> pd.DataFrame:
    """% de nulos por columna en la tabla de origen y en el dataset unido (left e inner).

    La columna intermedia — la tabla de origen restringida a las entidades que sí
    aparecen en opiniones — es la que separa las dos causas de nulos: los registros
    vacíos que nunca vamos a ver, y los que el merge sí arrastra al dataset.
    """
    con_opinion = {
        "libros": libros[libros[config.COL_ID_LIBRO].isin(set(opiniones[config.COL_ID_LIBRO]))],
        "lectores": lectores[lectores[config.COL_ID_LECTOR].isin(set(opiniones[config.COL_ID_LECTOR]))],
        "opiniones": opiniones,
    }
    origen = {"libros": libros, "lectores": lectores, "opiniones": opiniones}

    filas = []
    for tabla, columnas in [("opiniones", COLS_OPINIONES),
                            ("libros", COLS_LIBROS),
                            ("lectores", COLS_LECTORES)]:
        for col in columnas:
            col_origen = RENOMBRES.get(col, col)
            antes = 100 * origen[tabla][col_origen].isna().mean()
            filtrado = 100 * con_opinion[tabla][col_origen].isna().mean()
            fila = {
                "tabla": tabla,
                "columna": col,
                "% nulos ANTES (tabla completa)": round(antes, 2),
                "% nulos (sólo con opiniones)": round(filtrado, 2),
            }
            for nombre_how, df in unidos.items():
                fila[f"% nulos DESPUÉS ({nombre_how})"] = round(100 * df[col].isna().mean(), 2)
            fila["reducción (pp)"] = round(antes - 100 * unidos["left"][col].isna().mean(), 2)
            filas.append(fila)

    return pd.DataFrame(filas).set_index(["tabla", "columna"])


def resumen_nulos_libros(libros: pd.DataFrame, unidos: dict[str, pd.DataFrame]) -> None:
    """El número del informe: cuánto del 60% de nulos de `libros` nos toca limpiar."""
    cols_origen = [RENOMBRES.get(c, c) for c in COLS_LIBROS]

    celdas_tabla = libros[cols_origen].size
    nulas_tabla = int(libros[cols_origen].isna().sum().sum())

    # Un libro está "vacío" si no tiene ningún atributo cargado.
    vacios = libros[cols_origen].isna().all(axis=1)
    print(f"Celdas nulas en las columnas de `libros` (tabla completa): "
          f"{nulas_tabla:,} de {celdas_tabla:,}  ({100 * nulas_tabla / celdas_tabla:.2f} %)")
    print(f"Libros sin ningún atributo cargado (fila entera vacía): "
          f"{int(vacios.sum()):,} de {len(libros):,}  ({100 * vacios.mean():.2f} %)")

    for nombre_how, df in unidos.items():
        nulas = int(df[COLS_LIBROS].isna().sum().sum())
        print(f"Nulos en columnas de libros dentro del dataset unido ({nombre_how}): "
              f"{nulas:,} de {df[COLS_LIBROS].size:,}  ({100 * nulas / df[COLS_LIBROS].size:.2f} %)")


# --------------------------------------------------------------------------------------
# 3. left vs inner
# --------------------------------------------------------------------------------------

def comparar_how(unidos: dict[str, pd.DataFrame], opiniones: pd.DataFrame) -> pd.DataFrame:
    """Costo y beneficio de cada `how`, en filas y en calidad de las columnas."""
    filas = []
    for nombre_how, df in unidos.items():
        con_target = carga.construir_target(df)
        nulos_libros = 100 * df[COLS_LIBROS].isna().mean().mean()
        nulos_lectores = 100 * df[COLS_LECTORES].isna().mean().mean()
        filas.append({
            "how": nombre_how,
            "filas tras el merge": len(df),
            "filas tras quitar rating 6": len(con_target),
            "% de opiniones conservadas": round(100 * len(df) / len(opiniones), 3),
            "% nulos promedio (cols. libros)": round(nulos_libros, 2),
            "% nulos promedio (cols. lectores)": round(nulos_lectores, 2),
            "% clase 1 (gustó)": round(100 * con_target[config.TARGET].mean(), 2),
        })
    return pd.DataFrame(filas).set_index("how")


def target_en_huerfanas(base_left: pd.DataFrame) -> pd.DataFrame:
    """¿Las opiniones huérfanas tienen un target distinto? Si no, dropearlas no sesga."""
    df = carga.construir_target(base_left)
    es_huerfana = df["titulo"].isna() | df["nombre"].isna()
    resumen = (
        df.assign(grupo=es_huerfana.map({True: "huérfanas", False: "con libro y lector"}))
        .groupby("grupo")[config.TARGET]
        .agg(filas="size", **{"% gustó": lambda s: round(100 * s.mean(), 2)})
    )
    return resumen


# --------------------------------------------------------------------------------------
# 4. Distribución de clases
# --------------------------------------------------------------------------------------

def distribucion_clases(opiniones: pd.DataFrame, base: pd.DataFrame) -> None:
    print("Ratings en la tabla de opiniones (antes de construir el target):")
    conteo = opiniones[config.COL_RATING].value_counts().sort_index()
    tabla = pd.DataFrame({
        "opiniones": conteo,
        "% del total": (100 * conteo / len(opiniones)).round(2),
        "clase": [
            "1 (gustó)" if r >= config.RATING_MIN_GUSTO
            else "DESCARTADO" if r == config.RATING_DESCARTADO
            else "0 (no gustó)"
            for r in conteo.index
        ],
    })
    print(tabla.to_string())

    descartadas = int((opiniones[config.COL_RATING] == config.RATING_DESCARTADO).sum())
    print(f"\nFilas eliminadas por rating == 6: {descartadas:,} "
          f"({100 * descartadas / len(opiniones):.2f} % de las opiniones)")

    print(f"\nDistribución final de `{config.TARGET}` ({len(base):,} filas):")
    dist = base[config.TARGET].value_counts().sort_index()
    print(pd.DataFrame({
        "filas": dist,
        "%": (100 * dist / len(base)).round(2),
    }).to_string())
    ratio = dist.max() / dist.min()
    print(f"\nRatio de desbalanceo: {ratio:.2f} a 1 — desbalanceo clásico, no extremo.")


# --------------------------------------------------------------------------------------
# Runner
# --------------------------------------------------------------------------------------

def main(how: str = config.HOW_UNION) -> pd.DataFrame:
    pd.set_option("display.width", 200)
    pd.set_option("display.max_columns", 50)

    titulo("0. LECTURA DE LAS TRES TABLAS")
    print(f"libros:    {config.CSV_LIBROS}")
    print(f"lectores:  {config.CSV_LECTORES}")
    print(f"opiniones: {config.CSV_OPINIONES}")
    libros, lectores, opiniones = carga.cargar_tablas()

    titulo("1. PERFIL DE CADA TABLA POR SEPARADO")
    carga.perfilar_tablas(libros, lectores, opiniones)

    titulo("2. COBERTURA: CUÁNTO DE CADA TABLA USA EL DATASET")
    print(cobertura(libros, lectores, opiniones).to_string())
    print("\nOpiniones huérfanas (referencian un ID que no existe):")
    print(huerfanas(libros, lectores, opiniones).to_string())

    unidos = {
        "left": carga.unir(libros, lectores, opiniones, how="left"),
        "inner": carga.unir(libros, lectores, opiniones, how="inner"),
    }

    titulo("3. NULOS ANTES Y DESPUÉS DEL MERGE")
    nulos = tabla_nulos(libros, lectores, opiniones, unidos)
    print(nulos.to_string())

    titulo("4. EL 60% DE NULOS DE `libros` NO ES UN PROBLEMA A LIMPIAR")
    resumen_nulos_libros(libros, unidos)

    titulo("5. left vs inner")
    comparacion = comparar_how(unidos, opiniones)
    print(comparacion.to_string())
    print("\n¿Las opiniones huérfanas tienen otro comportamiento de target?")
    print(target_en_huerfanas(unidos["left"]).to_string())

    base = carga.construir_target(unidos[how])

    titulo(f"6. DISTRIBUCIÓN DE CLASES (dataset construido con how='{how}')")
    distribucion_clases(opiniones, base)

    titulo("7. SALIDAS")
    # Las dos tablas van al informe, así que se guardan en resultados/ (sí versionado).
    config.DIR_RESULTADOS.mkdir(parents=True, exist_ok=True)
    nulos.to_csv(config.DIR_RESULTADOS / "01_nulos_antes_despues.csv")
    comparacion.to_csv(config.DIR_RESULTADOS / "01_comparacion_left_inner.csv")
    print(f"Guardado: {config.DIR_RESULTADOS / '01_nulos_antes_despues.csv'}")
    print(f"Guardado: {config.DIR_RESULTADOS / '01_comparacion_left_inner.csv'}")

    config.DIR_CHECKPOINTS.mkdir(parents=True, exist_ok=True)
    base.to_pickle(config.CHECKPOINT_BASE)
    print(f"Guardado: {config.CHECKPOINT_BASE}")
    print(f"Dimensiones: {base.shape[0]:,} filas × {base.shape[1]} columnas  "
          f"(how='{how}', {config.CHECKPOINT_BASE.stat().st_size / 1e6:.1f} MB)")
    print(f"Columnas: {list(base.columns)}")
    return base


## 4. Ejecución

Corre el diagnóstico completo y deja el dataset base en `checkpoints/01_base.pkl`.


In [6]:
%%time
base = main()



0. LECTURA DE LAS TRES TABLAS
libros:    /home/user/What_Book_Do_I_Read/libros.csv
lectores:  /home/user/What_Book_Do_I_Read/lectores.csv
opiniones: /home/user/What_Book_Do_I_Read/interacciones.csv



1. PERFIL DE CADA TABLA POR SEPARADO

### Tabla `libros` — 128,743 filas × 9 columnas
             tipo  % nulos  valores únicos
id_libro      str     0.00          128743
titulo        str    60.76           48356
autor         str    60.76           21654
genero        str    60.76              65
editorial     str    60.76            2908
anio_edicion  str    60.76             118
isbn          str    60.76           49930
resumen       str    62.10           47991
img_src       str    60.76           47861
  IDs únicos en `id_libro`: 128,743
  Filas duplicadas por la clave ['id_libro']: 0

### Tabla `lectores` — 11,285 filas × 5 columnas


               tipo  % nulos  valores únicos
id_lector       str     0.00           11285
nombre          str     0.00            8830
genero          str     0.00               3
vive_en         str     4.73            1584
nacimiento  float64    30.39              94
  IDs únicos en `id_lector`: 11,285
  Filas duplicadas por la clave ['id_lector']: 0

### Tabla `opiniones` — 479,827 filas × 4 columnas


            tipo  % nulos  valores únicos
id_lector    str      0.0           10689
id_libro     str      0.0           50787
fecha        str      0.0            6619
rating     int64      0.0              10
  IDs únicos en `id_lector`: 10,689
  IDs únicos en `id_libro`: 50,787


  Filas duplicadas por la clave ['id_lector', 'id_libro']: 0

2. COBERTURA: CUÁNTO DE CADA TABLA USA EL DATASET
          en la tabla  con al menos una opinión  % usado  sin ninguna opinión  % descartado por el merge
tabla                                                                                                   
libros         128743                     50451    39.19                78292                      60.81
lectores        11285                     10683    94.67                  602                       5.33

Opiniones huérfanas (referencian un ID que no existe):


...

                         % nulos ANTES (tabla completa)  % nulos (sólo con opiniones)  % nulos DESPUÉS (left)  % nulos DESPUÉS (inner)  reducción (pp)
tabla     columna                                                                                                                                     
opiniones id_lector                                0.00                          0.00                    0.00                     0.00            0.00
          id_libro                                 0.00                          0.00                    0.00                     0.00            0.00
          fecha                                    0.00                          0.00                    0.00                     0.00            0.00
          rating                                   0.00                          0.00                    0.00                     0.00            0.00
libros    titulo                                  60.76                          0.01         

Nulos en columnas de libros dentro del dataset unido (left): 10,526 de 3,838,616  (0.27 %)
Nulos en columnas de libros dentro del dataset unido (inner): 4,005 de 3,830,960  (0.10 %)

5. left vs inner


       filas tras el merge  filas tras quitar rating 6  % de opiniones conservadas  % nulos promedio (cols. libros)  % nulos promedio (cols. lectores)  % clase 1 (gustó)
how                                                                                                                                                                      
left                479827                      389508                     100.000                             0.27                               8.12              83.88
inner               478870                      388720                      99.801                             0.10                               8.09              83.89

¿Las opiniones huérfanas tienen otro comportamiento de target?
                     filas  % gustó
grupo                              
con libro y lector  388716    83.89
huérfanas              792    75.00



6. DISTRIBUCIÓN DE CLASES (dataset construido con how='left')
Ratings en la tabla de opiniones (antes de construir el target):
        opiniones  % del total         clase
rating                                      
1            6461         1.35  0 (no gustó)
2            5385         1.12  0 (no gustó)
3            4899         1.02  0 (no gustó)
4           19440         4.05  0 (no gustó)
5           26623         5.55  0 (no gustó)
6           90319        18.82    DESCARTADO
7           89222        18.59     1 (gustó)
8          126427        26.35     1 (gustó)
9           55852        11.64     1 (gustó)
10          55199        11.50     1 (gustó)

Filas eliminadas por rating == 6: 90,319 (18.82 % de las opiniones)

Distribución final de `gusto` (389,508 filas):
        filas      %
gusto               
0       62808  16.12
1      326700  83.88

Ratio de desbalanceo: 5.20 a 1 — desbalanceo clásico, no extremo.

7. SALIDAS
Guardado: /home/user/What_Book_Do_I_Read/resultados/

Guardado: /home/user/What_Book_Do_I_Read/checkpoints/01_base.pkl
Dimensiones: 389,508 filas × 17 columnas  (how='left', 73.5 MB)
Columnas: ['id_lector', 'id_libro', 'fecha', 'rating', 'titulo', 'autor', 'genero_libro', 'editorial', 'anio_edicion', 'isbn', 'resumen', 'img_src', 'nombre', 'genero_lector', 'vive_en', 'nacimiento', 'gusto']
CPU times: user 5.7 s, sys: 3.14 s, total: 8.84 s
Wall time: 11.5 s
